In [ ]:
import pandas as pd
import os
from sqlalchemy import create_engine
from langchain_openai import ChatOpenAI
from langchain_community.agent_toolkits import create_sql_agent
from langchain_community.utilities import SQLDatabase

# --- STEP 1: DATA LOADING ---
f1_files = ["races.csv", "drivers.csv", "results.csv", "constructors.csv"]
db_path = "f1_history.db"
engine = create_engine(f"sqlite:///{db_path}")

for file in f1_files:
    if os.path.exists(file):
        table_name = file.replace(".csv", "")
        df = pd.read_csv(file)
        # Clean column names: remove dots, spaces, and lowercase everything
        df.columns = [c.strip().lower().replace(" ", "_").replace(".", "_") for c in df.columns]
        df.to_sql(table_name, engine, index=False, if_exists="replace")
    else:
        print(f"Warning: {file} not found. Please ensure it is in the current directory.")

print("F1 Database synchronized.")

F1 Database synchronized.


In [ ]:
llm = ChatOpenAI(
    api_key="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",
    base_url="https://openrouter.ai/api/v1",
    model="google/gemini-2.0-flash-001",
    default_headers={
        "HTTP-Referer": "http://localhost:3000",
        "X-Title": "F1 SQL Agent",
    },
    temperature=0 # Critical for SQL to ensure the agent doesn't "hallucinate" syntax
)

db = SQLDatabase.from_uri(f"sqlite:///{db_path}")

# Using "openai-tools" is correct for Gemini on OpenRouter.
f1_agent = create_sql_agent(
    llm=llm,
    db=db,
    agent_type="openai-tools",
    verbose=True
)

In [ ]:
try:
    print("\n--- Query 1: Total Races ---")
    res1 = f1_agent.invoke({"input": "How many races are in the database?"})
    print(res1["output"])

   
except Exception as e:
    print(f"An error occurred: {e}")


--- Query 1: Total Races ---


> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`
responded:  Finally, I should formulate a query to answer the question.


circuits, constructor_standings, constructors, driver_standings, drivers, lap_times, qualifying, races, results
Invoking: `sql_db_schema` with `{'table_names': 'races'}`
responded: The `races` table seems relevant. I should inspect its schema.



CREATE TABLE races (
	raceid BIGINT, 
	year BIGINT, 
	round BIGINT, 
	circuitid BIGINT, 
	name TEXT, 
	date TEXT, 
	time TEXT, 
	url TEXT, 
	fp1_date TEXT, 
	fp1_time TEXT, 
	fp2_date TEXT, 
	fp2_time TEXT, 
	fp3_date TEXT, 
	fp3_time TEXT, 
	quali_date TEXT, 
	quali_time TEXT, 
	sprint_date TEXT, 
	sprint_time TEXT
)

/*
3 rows from races table:
raceid	year	round	circuitid	name	date	time	url	fp1_date	fp1_time	fp2_date	fp2_time	fp3_date	fp3_time	quali_date	quali_time	sprint_date	sprint_time
1	2009	1	1	Australian Grand Prix	2009-03-29	06:00:00	http://en.wik

In [ ]:
try:
    print("\n--- Query 2: Top Drivers ---")
    query = "List the top 5 drivers with the most wins in F1 history and their total win count."
    res2 = f1_agent.invoke({"input": query})
    print(f"\nFinal Result: {res2['output']}")
except Exception as e:
    print(f"An error occurred: {e}")


--- Query 2: Top Drivers ---


> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`
responded:  Finally, I should construct a query to get the driver wins.


circuits, constructor_standings, constructors, driver_standings, drivers, lap_times, qualifying, races, results
Invoking: `sql_db_schema` with `{'table_names': 'drivers, driver_standings, races, results'}`
responded: Okay, the tables are circuits, constructor_standings, constructors, driver_standings, drivers, lap_times, qualifying, races, results.
The most relevant tables seem to be drivers, driver_standings, races, and results.



CREATE TABLE driver_standings (
	driverstandingsid BIGINT, 
	raceid BIGINT, 
	driverid BIGINT, 
	points FLOAT, 
	position BIGINT, 
	positiontext TEXT, 
	wins BIGINT
)

/*
3 rows from driver_standings table:
driverstandingsid	raceid	driverid	points	position	positiontext	wins
1	18	1	10.0	1	1	1
2	18	2	8.0	2	2	0
3	18	3	6.0	3	3	0
*/


CREATE TABLE drivers (
	driverid BIGINT,